In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from datetime import timedelta
import itertools
from tqdm import tqdm
import sys

sys.path.append('../data/NACC/')
from data_cleaning_util import *

In [ ]:
uds_path = "C:/Users/liang/Documents/GitHub/fusion/data/NACC/raw_data/investigator_nacc64.csv"
mriqc_path = "C:/Users/liang/Documents/GitHub/fusion/data/NACC/raw_data/investigator_scan_mri_nacc66/investigator_scan_mriqc_nacc66.csv"
mrisbm_path="C:/Users/liang/Documents/GitHub/fusion/data/NACC/raw_data/investigator_scan_mri_nacc66/investigator_scan_mrisbm_nacc66.csv"

In [ ]:
uds_raw = uds_data = pd.read_csv(uds_path, low_memory=False)
mriqc_raw = mriqc_data = pd.read_csv(mriqc_path)
mrisbm_raw = mrisbm_data = pd.read_csv(mrisbm_path)

In [ ]:
def find_matches(uds_data, mrisbm_data):
    # Convert VISITMO, VISITDAY, and VISITYR to integers if they are not already
    uds_data['VISITMO'] = uds_data['VISITMO'].astype(int)
    uds_data['VISITDAY'] = uds_data['VISITDAY'].astype(int)
    uds_data['VISITYR'] = uds_data['VISITYR'].astype(int)
    
    # Now convert VISITMO, VISITDAY, VISITYR to a single datetime column 'VISITDT'
    uds_data['VISITDT'] = pd.to_datetime(uds_data[['VISITYR', 'VISITMO', 'VISITDAY']].rename(
                                         columns={"VISITYR":'year', 'VISITMO':'month', 'VISITDAY':'day'}))
    
    # Convert SCANDT to datetime in mrisbm_data
    mrisbm_data['SCANDT'] = pd.to_datetime(mrisbm_data['SCANDT'])

    # Create an empty list to store the matches
    matches = []
    
    # Loop through each scan in the mrisbm_data
    for index, scan in mrisbm_data.iterrows():
        subject_id = scan['NACCID']
        scan_date = scan['SCANDT']
    
        # Filter uds_data for the same subject
        subject_visits = uds_data[uds_data['NACCID'] == subject_id].copy()  # Use .copy() to avoid SettingWithCopyWarning
        
        # Calculate the time difference between each visit and the scan
        subject_visits.loc[:, 'TIME_DIFF'] = (subject_visits['VISITDT'] - scan_date).abs()
    
        # Filter visits that are within 18 months (18 * 30 = 540 days)
        eligible_visits = subject_visits[(subject_visits['VISITDT'] <= scan_date) & 
                                         (scan_date - subject_visits['VISITDT'] <= timedelta(days=540))]
    
        # If there are eligible visits, find the closest visit
        if not eligible_visits.empty:
            closest_visit = eligible_visits.loc[eligible_visits['TIME_DIFF'].idxmin()]
            matches.append((closest_visit, scan))
    
    # Create new DataFrames for the filtered matched visits and scans
    matched_visits = pd.DataFrame([match[0] for match in matches])
    matched_scans = pd.DataFrame([match[1] for match in matches])
    
    # Ensure the rows are ordered based on subject ID and visit/scan dates
    matched_visits = matched_visits.sort_values(by=['NACCID', 'VISITDT'])
    matched_scans = matched_scans.sort_values(by=['NACCID', 'SCANDT'])
    
    # Reset index for clarity
    matched_visits = matched_visits.reset_index(drop=True)
    matched_scans = matched_scans.reset_index(drop=True)

    return matched_visits, matched_scans

In [ ]:
# Find matched mri and uds data
uds_matched_raw, mrisbm_matched_raw = find_matches(uds_data, mrisbm_data)

In [ ]:
# drop variables in uds data that has more than 10% missing data
missing_proportion = uds_matched_raw.isna().mean()
uds_matched = uds_matched_raw.loc[:, missing_proportion <= 0.1]

# Preprocess the mri data: drop columns irrelavant to prediction such as image ID and descriptions
columns_to_drop = ['NACCID', 'NACCADC', 'SCANDT', 'LONI_IMAGE_FLAIR', 'DESCRIPTION_FLAIR', 'LONI_IMAGE_T1', 'DESCRIPTION_T1', 'FREESURFER_VERSION']

# Matched mri data still contains missing values, imputation will be done after train test splitting
mrisbm_matched = mrisbm_matched_raw.drop(columns=columns_to_drop)

In [ ]:
uds_matched.shape

## Preprocessing Version 1
Two modalities: UDS and MRI

Extract features based on Yueqi's preprocessing

In [ ]:
# Preprocess uds data: select and encode relevant features
cat_feats, ord_feats, num_feats, label_feat, cdr_feats = get_feature_types(uds_matched)
all_uds_feats = cat_feats+ord_feats+num_feats+label_feat
uds_matched = uds_matched[all_uds_feats]

In [ ]:
uds_matched.shape

In [ ]:
# Encoded uds data still contains missing values, imputation will be done after train test splitting
ordinal_preprocessor = encode_feature_types(uds_matched)
encoded_uds = pd.DataFrame(ordinal_preprocessor.transform(uds_matched),columns=ordinal_preprocessor.get_feature_names_out(),index=uds_matched.index.values)

In [ ]:
# Save data version 1 (2 totoal modalities)
data_dir = "C:/Users/liang/Documents/GitHub/fusion/data/NACC/preprocessed_v1/"

encoded_uds.to_csv(data_dir+'uds_matched.csv', index=False)
mrisbm_matched.to_csv(data_dir+'mrisbm_matched.csv', index=False)

## Preprocessing Version 2
Three modalities: UDS derived history modality, UDS derived survey modality and MRI.

Extract features based on Yueqi's preprocessing.

In [ ]:
# Preprocess uds data: select and encode relevant features
cat_feats, ord_feats, num_feats, label_feat, cdr_feats = get_feature_types(uds_matched)
all_uds_feats = cat_feats+ord_feats+num_feats+label_feat
uds_matched = uds_matched[all_uds_feats]

In [ ]:
# Encoded uds data still contains missing values, imputation will be done after train test splitting
ordinal_preprocessor = encode_feature_types(uds_matched)
encoded_uds = pd.DataFrame(ordinal_preprocessor.transform(uds_matched),columns=ordinal_preprocessor.get_feature_names_out(),index=uds_matched.index.values)

In [ ]:
mod_history, mod_survey, label_feat = get_modality_features(encoded_uds)

In [ ]:
uds_history = encoded_uds[mod_history]
uds_survey = encoded_uds[mod_survey]
label = encoded_uds[label_feat]

In [ ]:
# Save data version 2 (3 total modalities)
data_dir = "C:/Users/liang/Documents/GitHub/fusion/data/NACC/preprocessed_v2/"

uds_history.to_csv(data_dir+'uds_history.csv', index=False)
uds_survey.to_csv(data_dir+'uds_survey.csv', index=False)
mrisbm_matched.to_csv(data_dir+'mrisbm.csv', index=False)
label.to_csv(data_dir+'label.csv', index=False)

## Preprocessing Version 3
Four modalities: UDS derived history modality, UDS derived survey modality and UDS derived clinical/testing modalities and MRI.

In [ ]:
# Preprocess uds data: select and encode relevant features
cat_feats, ord_feats, num_feats, label_feat, cdr_feats = get_feature_types(uds_matched)
all_uds_feats = cat_feats+ord_feats+num_feats+label_feat
uds_matched = uds_matched[all_uds_feats]

In [ ]:
# Encoded uds data still contains missing values, imputation will be done after train test splitting
ordinal_preprocessor = encode_feature_types(uds_matched)
encoded_uds = pd.DataFrame(ordinal_preprocessor.transform(uds_matched),columns=ordinal_preprocessor.get_feature_names_out(),index=uds_matched.index.values)

In [ ]:
mod_history, mod_survey, mod_testing, label_feat = get_modality_features(encoded_uds, fine_grained=True)

In [ ]:
uds_history = encoded_uds[mod_history]
uds_survey = encoded_uds[mod_survey]
uds_testing = encoded_uds[mod_testing]
label = encoded_uds[label_feat]

In [ ]:
# Save data version 3 (4 total modalities)
data_dir = "C:/Users/liang/Documents/GitHub/fusion/data/NACC/preprocessed_v3/"

uds_history.to_csv(data_dir+'uds_history.csv', index=False)
uds_survey.to_csv(data_dir+'uds_survey.csv', index=False)
uds_testing.to_csv(data_dir+'uds_testing.csv', index=False)
mrisbm_matched.to_csv(data_dir+'mrisbm.csv', index=False)
label.to_csv(data_dir+'label.csv', index=False)

## Zahra Preprocess

In [3]:
# ============================================================
# PREPROCESSING VERSION 3 FOR JOINT MULTIMODAL METHOD
# KEEP RAW MODALITY WIDTHS: 66 / 39 / 29
# ------------------------------------------------------------
# Key design:
#   - split UDS into modalities FIRST
#   - DO NOT one-hot encode
#   - each original variable stays as exactly one column
#   - categorical / ordinal variables are converted to one numeric code each
#   - numeric variables are median-imputed and standardized
#   - categorical / ordinal variables are mode-imputed
#   - MRI is matched to each UDS visit within 18 months
#   - unmatched MRI rows are kept and filled with MISSING_VALUE
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# ============================================================
# PATHS
# ============================================================

RAW_DIR = "./raw_data"
OUT_DIR = "./preprocessed_v3_for_joint_missingMRI_keepRawDims"
os.makedirs(OUT_DIR, exist_ok=True)

uds_path = os.path.join(RAW_DIR, "investigator_nacc64.csv")
mrisbm_path = os.path.join(
    RAW_DIR,
    "investigator_scan_mri_nacc66",
    "investigator_scan_mrisbm_nacc66.csv"
)

MISSING_VALUE = -999.0
MRI_WINDOW_DAYS = 18 * 30  # approximate 18 months


# ============================================================
# FEATURE DEFINITIONS
# ============================================================

def get_feature_types(df, tier2=True):
    cat_demo_feats = ['NACCNIHR','PRIMLANG','SEX','HISPANIC']
    ord_demo_feats = ['MARISTAT','NACCLIVS','INDEPEND','RESIDENC']
    num_demo_feats = ['NACCAGE','EDUC']

    cat_phist_feats = ['TOBAC30', 'TOBAC100','NACCTBI','DEP2YRS', 'DEPOTHR',
            'ANYMEDS','NACCAAAS', 'NACCAANX', 'NACCAC', 'NACCACEI',
            'NACCADEP', 'NACCAHTN', 'NACCANGI', 'NACCAPSY',
            'NACCBETA', 'NACCCCBS', 'NACCDBMD', 'NACCDIUR', 'NACCEMD',
            'NACCEPMD', 'NACCHTNC', 'NACCLIPL', 'NACCNSD', 'NACCPDMD',
            'NACCVASD']
    ord_phist_feats = ['NACCAMD','PACKSPER','CVHATT', 'CVAFIB', 'CVANGIO', 'CVBYPASS', 'CBTIA',
            'CVPACE', 'CVCHF', 'CVOTHR', 'CBSTROKE','SEIZURES','NCOTHR',
            'DIABETES','HYPERTEN', 'HYPERCHO', 'B12DEF','THYROID', 'INCONTU',
            'INCONTF','ALCOHOL', 'ABUSOTHR','PSYCDIS']
    num_phist_feats = ['SMOKYRS','NACCSTYR','NACCTIYR']

    cat_fhist_feats = ['NACCFADM', 'NACCFFTD']
    ord_fhist_feats = ['NACCFAM', 'NACCMOM', 'NACCDAD']

    cat_phys_feats = ['NACCNREX','FOCLSYM','FOCLSIGN']
    ord_phys_feats = ['DECSUB','VISION', 'VISCORR','VISWCORR','HEARING', 'HEARAID', 'HEARWAID']
    num_phys_feats = ['HEIGHT', 'WEIGHT','BPSYS', 'BPDIAS', 'HRATE','NACCBMI']

    ord_gds_feats = ['NOGDS','SATIS', 'DROPACT', 'EMPTY', 'BORED', 'SPIRITS', 'AFRAID',
            'HAPPY', 'HELPLESS', 'STAYHOME', 'MEMPROB', 'WONDRFUL', 'WRTHLESS',
            'ENERGY', 'HOPELESS', 'BETTER','NACCGDS']
    ord_faq_feats = ['BILLS', 'TAXES','SHOPPING', 'GAMES', 'STOVE',
            'MEALPREP', 'EVENTS', 'PAYATTN','REMDATES', 'TRAVEL']
    ord_npi_feats = ['DELSEV', 'HALLSEV', 'AGITSEV', 'DEPDSEV', 'ANXSEV',
                'ELATSEV', 'APASEV', 'DISNSEV', 'IRRSEV', 'MOTSEV', 'NITESEV',
                'APPSEV']

    ord_np_feats = ['MMSEORDA','MMSEORLO']

    label_feat = ['NACCUDSD']
    cdr_feats = ['MEMORY', 'ORIENT', 'JUDGMENT', 'COMMUN', 'HOMEHOBB',
                 'PERSCARE', 'CDRSUM', 'CDRGLOB']

    cat_feats = cat_demo_feats + cat_phist_feats + cat_fhist_feats + cat_phys_feats

    if tier2:
        num_np_feats = ['NACCMMSE','MEMUNITS','DIGIF', 'DIGIFLEN', 'DIGIB', 'DIGIBLEN',
                        'ANIMALS', 'VEG','BOSTON', 'TRAILA', 'TRAILB']
    else:
        num_np_feats = ['NACCMMSE']

    ord_feats = list(ord_demo_feats + ord_phist_feats + ord_fhist_feats + ord_phys_feats +
                     ord_npi_feats + ord_gds_feats + ord_faq_feats + ord_np_feats)
    num_feats = num_demo_feats + num_phist_feats + num_phys_feats + num_np_feats

    cat_feats = [x for x in cat_feats if x in df.columns]
    ord_feats = [x for x in ord_feats if x in df.columns]
    num_feats = [x for x in num_feats if x in df.columns]
    label_feat = [x for x in label_feat if x in df.columns]
    cdr_feats = [x for x in cdr_feats if x in df.columns]

    return cat_feats, ord_feats, num_feats, label_feat, cdr_feats


def get_modality_features(df, tier2=True, fine_grained=False):
    cat_demo_feats = ['NACCNIHR','PRIMLANG','SEX','HISPANIC']
    ord_demo_feats = ['MARISTAT','NACCLIVS','INDEPEND','RESIDENC']
    num_demo_feats = ['NACCAGE','EDUC']

    cat_phist_feats = ['TOBAC30', 'TOBAC100','NACCTBI','DEP2YRS', 'DEPOTHR',
            'ANYMEDS','NACCAAAS', 'NACCAANX', 'NACCAC', 'NACCACEI',
            'NACCADEP', 'NACCAHTN', 'NACCANGI', 'NACCAPSY',
            'NACCBETA', 'NACCCCBS', 'NACCDBMD', 'NACCDIUR', 'NACCEMD',
            'NACCEPMD', 'NACCHTNC', 'NACCLIPL', 'NACCNSD', 'NACCPDMD',
            'NACCVASD']
    ord_phist_feats = ['NACCAMD','PACKSPER','CVHATT', 'CVAFIB', 'CVANGIO', 'CVBYPASS', 'CBTIA',
            'CVPACE', 'CVCHF', 'CVOTHR', 'CBSTROKE','SEIZURES','NCOTHR',
            'DIABETES','HYPERTEN', 'HYPERCHO', 'B12DEF','THYROID', 'INCONTU',
            'INCONTF','ALCOHOL', 'ABUSOTHR','PSYCDIS']
    num_phist_feats = ['SMOKYRS','NACCSTYR','NACCTIYR']

    cat_fhist_feats = ['NACCFADM', 'NACCFFTD']
    ord_fhist_feats = ['NACCFAM', 'NACCMOM', 'NACCDAD']

    cat_phys_feats = ['NACCNREX','FOCLSYM','FOCLSIGN']
    ord_phys_feats = ['DECSUB','VISION', 'VISCORR','VISWCORR','HEARING', 'HEARAID', 'HEARWAID']
    num_phys_feats = ['HEIGHT', 'WEIGHT','BPSYS', 'BPDIAS', 'HRATE','NACCBMI']

    ord_gds_feats = ['NOGDS','SATIS', 'DROPACT', 'EMPTY', 'BORED', 'SPIRITS', 'AFRAID',
            'HAPPY', 'HELPLESS', 'STAYHOME', 'MEMPROB', 'WONDRFUL', 'WRTHLESS',
            'ENERGY', 'HOPELESS', 'BETTER','NACCGDS']
    ord_faq_feats = ['BILLS', 'TAXES','SHOPPING', 'GAMES', 'STOVE',
            'MEALPREP', 'EVENTS', 'PAYATTN','REMDATES', 'TRAVEL']
    ord_npi_feats = ['DELSEV', 'HALLSEV', 'AGITSEV', 'DEPDSEV', 'ANXSEV',
                'ELATSEV', 'APASEV', 'DISNSEV', 'IRRSEV', 'MOTSEV', 'NITESEV',
                'APPSEV']

    ord_np_feats = ['MMSEORDA','MMSEORLO']
    if tier2:
        num_np_feats = ['NACCMMSE','MEMUNITS','DIGIF', 'DIGIFLEN', 'DIGIB', 'DIGIBLEN',
                        'ANIMALS', 'VEG','BOSTON', 'TRAILA', 'TRAILB']
    else:
        num_np_feats = ['NACCMMSE']

    label_feat = ['NACCUDSD']

    demo_feats = cat_demo_feats + ord_demo_feats + num_demo_feats
    phist_feats = cat_phist_feats + ord_phist_feats + num_phist_feats
    fhist_feats = cat_fhist_feats + ord_fhist_feats

    mod_history = [x for x in (demo_feats + phist_feats + fhist_feats) if x in df.columns]

    phys_feats = cat_phys_feats + ord_phys_feats + num_phys_feats
    npi_feats = ord_gds_feats + ord_faq_feats + ord_npi_feats
    npt_feats = ord_np_feats + num_np_feats

    if fine_grained:
        mod_survey = [x for x in npi_feats if x in df.columns]
        mod_testing = [x for x in (phys_feats + npt_feats) if x in df.columns]
        return mod_history, mod_survey, mod_testing, label_feat
    else:
        mod_survey = [x for x in (phys_feats + npi_feats + npt_feats) if x in df.columns]
        return mod_history, mod_survey, label_feat


# ============================================================
# MODALITY PREPROCESSING WITHOUT ONE-HOT
# ============================================================

def preprocess_one_modality_keep_width(df_modality):
    """
    Keeps exactly one output column per original input column.
    No one-hot encoding.
    """
    x = df_modality.copy()

    cat_feats, ord_feats, num_feats, _, _ = get_feature_types(x, tier2=True)
    cat_feats = [c for c in cat_feats if c in x.columns]
    ord_feats = [c for c in ord_feats if c in x.columns]
    num_feats = [c for c in num_feats if c in x.columns]

    out = pd.DataFrame(index=x.index)

    # categorical: mode impute + ordinal encode to ONE column each
    if len(cat_feats) > 0:
        x_cat = x[cat_feats].copy()
        cat_imputer = SimpleImputer(strategy='most_frequent')
        x_cat_imp = pd.DataFrame(cat_imputer.fit_transform(x_cat), columns=cat_feats, index=x.index)

        cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        x_cat_enc = pd.DataFrame(cat_encoder.fit_transform(x_cat_imp), columns=cat_feats, index=x.index)
        out = pd.concat([out, x_cat_enc], axis=1)

    # ordinal: mode impute, keep one column each
    if len(ord_feats) > 0:
        x_ord = x[ord_feats].copy()
        for c in ord_feats:
            x_ord[c] = pd.to_numeric(x_ord[c], errors='coerce')
        ord_imputer = SimpleImputer(strategy='most_frequent')
        x_ord_imp = pd.DataFrame(ord_imputer.fit_transform(x_ord), columns=ord_feats, index=x.index)
        out = pd.concat([out, x_ord_imp], axis=1)

    # numeric: median impute + z-score, one column each
    if len(num_feats) > 0:
        x_num = x[num_feats].copy()
        for c in num_feats:
            x_num[c] = pd.to_numeric(x_num[c], errors='coerce')
        num_imputer = SimpleImputer(strategy='median')
        x_num_imp = pd.DataFrame(num_imputer.fit_transform(x_num), columns=num_feats, index=x.index)

        scaler = StandardScaler()
        x_num_scl = pd.DataFrame(scaler.fit_transform(x_num_imp), columns=num_feats, index=x.index)
        out = pd.concat([out, x_num_scl], axis=1)

    # preserve original column order
    out = out[[c for c in x.columns if c in out.columns]].astype(np.float32)
    return out


def preprocess_mri_full(df_mri):
    x = df_mri.copy()
    for c in x.columns:
        x[c] = pd.to_numeric(x[c], errors='coerce')

    for c in x.columns:
        med = x[c].median()
        if pd.isna(med):
            med = 0.0
        x[c] = x[c].fillna(med)

    mu = x.mean(axis=0)
    sd = x.std(axis=0).replace(0, 1.0)
    x = ((x - mu) / sd).astype(np.float32)
    return x


def build_label_from_naccudsd(series):
    s = pd.to_numeric(series, errors='coerce')
    keep = s.isin([1, 2, 3, 4])

    out = pd.Series(index=series.index, dtype='float')
    out.loc[keep & s.isin([1, 2])] = 0
    out.loc[keep & (s == 3)] = 1
    out.loc[keep & (s == 4)] = 2
    return out, keep


def choose_id_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"None of these ID columns were found: {candidates}")


def build_uds_datetime(df):
    if all(c in df.columns for c in ["VISITYR", "VISITMO", "VISITDAY"]):
        yy = pd.to_numeric(df["VISITYR"], errors="coerce")
        mm = pd.to_numeric(df["VISITMO"], errors="coerce")
        dd = pd.to_numeric(df["VISITDAY"], errors="coerce")
        return pd.to_datetime(dict(year=yy, month=mm, day=dd), errors="coerce")
    raise ValueError("Could not build UDS date; expected VISITYR/VISITMO/VISITDAY.")


def find_best_mri_match_for_each_uds(uds_df, mri_df, id_col, uds_date_series, mri_date_col, max_days=MRI_WINDOW_DAYS):
    mri_dates = pd.to_datetime(mri_df[mri_date_col], errors='coerce')
    mri_df_local = mri_df.copy()
    mri_df_local["_MRI_DATE_"] = mri_dates

    match_idx = []

    for i, row in uds_df.iterrows():
        sid = row[id_col]
        uds_date = uds_date_series.loc[i]

        cand = mri_df_local[mri_df_local[id_col] == sid].copy()
        if len(cand) == 0 or pd.isna(uds_date):
            match_idx.append(np.nan)
            continue

        cand["_ABS_DAYS_"] = (cand["_MRI_DATE_"] - uds_date).abs().dt.days
        cand = cand[cand["_ABS_DAYS_"] <= max_days]

        if len(cand) == 0:
            match_idx.append(np.nan)
            continue

        best = cand.sort_values("_ABS_DAYS_").index[0]
        match_idx.append(best)

    return pd.Series(match_idx, index=uds_df.index)


# ============================================================
# LOAD RAW DATA
# ============================================================

uds_raw = pd.read_csv(uds_path, low_memory=False)
mri_raw = pd.read_csv(mrisbm_path, low_memory=False)

print("Raw UDS shape :", uds_raw.shape)
print("Raw MRI shape :", mri_raw.shape)

ID_CANDIDATES = ["NACCID", "RID", "ID"]
MRI_DATE_CANDIDATES = ["SCANDT", "MRIDATE", "EXAMDATE"]

id_col = choose_id_column(uds_raw, ID_CANDIDATES)
mri_id_col = choose_id_column(mri_raw, ID_CANDIDATES)
if mri_id_col != id_col:
    mri_raw = mri_raw.rename(columns={mri_id_col: id_col})

mri_date_col = None
for c in MRI_DATE_CANDIDATES:
    if c in mri_raw.columns:
        mri_date_col = c
        break
if mri_date_col is None:
    raise ValueError(f"Could not find MRI date column among {MRI_DATE_CANDIDATES}")

uds_visit_date = build_uds_datetime(uds_raw)

print("Using ID column    :", id_col)
print("Using MRI date col :", mri_date_col)


# ============================================================
# LABEL + ELIGIBLE UDS VISITS
# ============================================================

diag_label, keep_diag = build_label_from_naccudsd(uds_raw["NACCUDSD"])
uds_keep = uds_raw.loc[keep_diag].copy().reset_index(drop=True)
uds_keep["LABEL"] = diag_label.loc[keep_diag].astype(int).values
uds_keep["_UDS_DATE_"] = uds_visit_date.loc[keep_diag].reset_index(drop=True)

print("Eligible UDS visits after diagnosis filtering:", uds_keep.shape[0])


# ============================================================
# SPLIT RAW UDS INTO 3 RAW MODALITIES FIRST
# ============================================================

mod_history, mod_survey, mod_testing, _ = get_modality_features(uds_keep, fine_grained=True)

uds_history_raw = uds_keep[mod_history].copy()
uds_survey_raw = uds_keep[mod_survey].copy()
uds_testing_raw = uds_keep[mod_testing].copy()

print("Raw modality dimensions before preprocessing:")
print("  history :", uds_history_raw.shape)
print("  survey  :", uds_survey_raw.shape)
print("  testing :", uds_testing_raw.shape)

uds_history = preprocess_one_modality_keep_width(uds_history_raw)
uds_survey = preprocess_one_modality_keep_width(uds_survey_raw)
uds_testing = preprocess_one_modality_keep_width(uds_testing_raw)

print("\nProcessed modality dimensions after KEEP-WIDTH preprocessing:")
print("  history :", uds_history.shape)
print("  survey  :", uds_survey.shape)
print("  testing :", uds_testing.shape)


# ============================================================
# MRI MATCHING AT UDS-VISIT LEVEL
# ============================================================

uds_meta = uds_keep[[id_col, "LABEL", "_UDS_DATE_"]].copy()

match_idx = find_best_mri_match_for_each_uds(
    uds_df=uds_meta,
    mri_df=mri_raw,
    id_col=id_col,
    uds_date_series=uds_meta["_UDS_DATE_"],
    mri_date_col=mri_date_col,
    max_days=MRI_WINDOW_DAYS
)

mri_available_mask = match_idx.notna().astype(int)

mri_feature_cols = [c for c in mri_raw.columns if c not in [id_col, mri_date_col]]
mri_feature_table = preprocess_mri_full(mri_raw[mri_feature_cols].copy())

aligned_mri_rows = []
for idx_match in match_idx:
    if pd.isna(idx_match):
        aligned_mri_rows.append(np.full(len(mri_feature_cols), MISSING_VALUE, dtype=np.float32))
    else:
        aligned_mri_rows.append(mri_feature_table.loc[int(idx_match)].values.astype(np.float32))

mrisbm_aligned = pd.DataFrame(aligned_mri_rows, columns=mri_feature_cols, index=uds_meta.index)

print("\nAligned MRI shape   :", mrisbm_aligned.shape)
print("MRI available count :", int(mri_available_mask.sum()))
print("MRI missing count   :", int((1 - mri_available_mask).sum()))


# ============================================================
# SAVE
# ============================================================

subject_id_out = uds_keep[[id_col]].copy().reset_index(drop=True)
visit_meta_out = uds_keep[[id_col, "_UDS_DATE_"]].copy().reset_index(drop=True)
label_out = uds_keep[["LABEL"]].copy().reset_index(drop=True)
mask_out = pd.DataFrame({"MRI_AVAILABLE": mri_available_mask.values.astype(int)})

uds_history = uds_history.reset_index(drop=True)
uds_survey = uds_survey.reset_index(drop=True)
uds_testing = uds_testing.reset_index(drop=True)
mrisbm_aligned = mrisbm_aligned.reset_index(drop=True)

subject_id_out.to_csv(os.path.join(OUT_DIR, "subject_id.csv"), index=False)
visit_meta_out.to_csv(os.path.join(OUT_DIR, "visit_meta.csv"), index=False)
label_out.to_csv(os.path.join(OUT_DIR, "label.csv"), index=False)
mask_out.to_csv(os.path.join(OUT_DIR, "mri_available_mask.csv"), index=False)

uds_history.to_csv(os.path.join(OUT_DIR, "uds_history.csv"), index=False)
uds_survey.to_csv(os.path.join(OUT_DIR, "uds_survey.csv"), index=False)
uds_testing.to_csv(os.path.join(OUT_DIR, "uds_testing.csv"), index=False)
mrisbm_aligned.to_csv(os.path.join(OUT_DIR, "mrisbm.csv"), index=False)

print("\nSaved files to:", OUT_DIR)
print("Files:")
print("  subject_id.csv")
print("  visit_meta.csv")
print("  label.csv")
print("  mri_available_mask.csv")
print("  uds_history.csv")
print("  uds_survey.csv")
print("  uds_testing.csv")
print("  mrisbm.csv")

Raw UDS shape : (185831, 1024)
Raw MRI shape : (1652, 200)
Using ID column    : NACCID
Using MRI date col : SCANDT
Eligible UDS visits after diagnosis filtering: 185831
Raw modality dimensions before preprocessing:
  history : (185831, 66)
  survey  : (185831, 39)
  testing : (185831, 29)

Processed modality dimensions after KEEP-WIDTH preprocessing:
  history : (185831, 66)
  survey  : (185831, 39)
  testing : (185831, 29)


/home/zmoslemi/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/zmoslemi/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/zmoslemi/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)



Aligned MRI shape   : (185831, 198)
MRI available count : 2379
MRI missing count   : 183452

Saved files to: ./preprocessed_v3_for_joint_missingMRI_keepRawDims
Files:
  subject_id.csv
  visit_meta.csv
  label.csv
  mri_available_mask.csv
  uds_history.csv
  uds_survey.csv
  uds_testing.csv
  mrisbm.csv


first visit

In [9]:
# ============================================================
# PREPROCESSING VERSION 4 — FIRST VISIT ONLY
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# ============================================================
# PATHS
# ============================================================

RAW_DIR = "./raw_data"
OUT_DIR = "./preprocessed_v4_firstvisit_missingMRI_keepRawDims"   # ← new dir
os.makedirs(OUT_DIR, exist_ok=True)

uds_path = os.path.join(RAW_DIR, "investigator_nacc64.csv")
mrisbm_path = os.path.join(
    RAW_DIR,
    "investigator_scan_mri_nacc66",
    "investigator_scan_mrisbm_nacc66.csv"
)

MISSING_VALUE = -999.0
MRI_WINDOW_DAYS = 10000 * 30


# ============================================================
# ALL HELPER FUNCTIONS — UNCHANGED
# (get_feature_types, get_modality_features,
#  preprocess_one_modality_keep_width, preprocess_mri_full,
#  build_label_from_naccudsd, choose_id_column,
#  build_uds_datetime, find_best_mri_match_for_each_uds)
# ============================================================

# ... paste all your existing helper functions here unchanged ...


# ============================================================
# LOAD RAW DATA — UNCHANGED
# ============================================================

uds_raw = pd.read_csv(uds_path, low_memory=False)
mri_raw = pd.read_csv(mrisbm_path, low_memory=False)

print("Raw UDS shape :", uds_raw.shape)
print("Raw MRI shape :", mri_raw.shape)

ID_CANDIDATES = ["NACCID", "RID", "ID"]
MRI_DATE_CANDIDATES = ["SCANDT", "MRIDATE", "EXAMDATE"]

id_col = choose_id_column(uds_raw, ID_CANDIDATES)
mri_id_col = choose_id_column(mri_raw, ID_CANDIDATES)
if mri_id_col != id_col:
    mri_raw = mri_raw.rename(columns={mri_id_col: id_col})

mri_date_col = None
for c in MRI_DATE_CANDIDATES:
    if c in mri_raw.columns:
        mri_date_col = c
        break
if mri_date_col is None:
    raise ValueError(f"Could not find MRI date column among {MRI_DATE_CANDIDATES}")

uds_visit_date = build_uds_datetime(uds_raw)

print("Using ID column    :", id_col)
print("Using MRI date col :", mri_date_col)


# ============================================================
# LABEL + ELIGIBLE UDS VISITS — UNCHANGED
# ============================================================

diag_label, keep_diag = build_label_from_naccudsd(uds_raw["NACCUDSD"])
uds_keep = uds_raw.loc[keep_diag].copy().reset_index(drop=True)
uds_keep["LABEL"] = diag_label.loc[keep_diag].astype(int).values
uds_keep["_UDS_DATE_"] = uds_visit_date.loc[keep_diag].reset_index(drop=True)

print("Eligible UDS visits after diagnosis filtering:", uds_keep.shape[0])


# ============================================================
# *** FIRST-VISIT FILTER — NEW BLOCK ***
# ============================================================

# Sort chronologically per subject so earliest visit is row-first,
# then drop all subsequent visits keeping only the first.
# Safe because _UDS_DATE_ is built from VISITYR/VISITMO/VISITDAY
# which are always present in the raw NACC data.

uds_keep = (
    uds_keep
    .sort_values(by=[id_col, "_UDS_DATE_"], ascending=True, na_position="last")
    .drop_duplicates(subset=[id_col], keep="first")
    .reset_index(drop=True)
)

# Rebuild _UDS_DATE_ cleanly after reset_index to avoid any
# index misalignment in downstream code
uds_keep["_UDS_DATE_"] = pd.to_datetime(
    uds_keep[["VISITYR", "VISITMO", "VISITDAY"]].rename(
        columns={"VISITYR": "year", "VISITMO": "month", "VISITDAY": "day"}
    ), errors="coerce"
)

print(f"After first-visit filter : {len(uds_keep)} rows "
      f"({uds_keep[id_col].nunique()} unique subjects)")

# Hard assertion — must be zero or something is wrong
assert uds_keep[id_col].duplicated().sum() == 0, \
    "BUG: duplicate subjects remain after first-visit filter"

# ============================================================
# EVERYTHING BELOW IS IDENTICAL TO v3 — NO CHANGES NEEDED
# ============================================================

mod_history, mod_survey, mod_testing, _ = get_modality_features(
    uds_keep, fine_grained=True)

uds_history_raw = uds_keep[mod_history].copy()
uds_survey_raw  = uds_keep[mod_survey].copy()
uds_testing_raw = uds_keep[mod_testing].copy()

print("Raw modality dimensions before preprocessing:")
print("  history :", uds_history_raw.shape)
print("  survey  :", uds_survey_raw.shape)
print("  testing :", uds_testing_raw.shape)

uds_history = preprocess_one_modality_keep_width(uds_history_raw)
uds_survey  = preprocess_one_modality_keep_width(uds_survey_raw)
uds_testing = preprocess_one_modality_keep_width(uds_testing_raw)

print("\nProcessed modality dimensions after KEEP-WIDTH preprocessing:")
print("  history :", uds_history.shape)
print("  survey  :", uds_survey.shape)
print("  testing :", uds_testing.shape)


# ============================================================
# MRI MATCHING — UNCHANGED
# (now matches at most 1 MRI per subject since UDS has 1 row each)
# ============================================================

uds_meta = uds_keep[[id_col, "LABEL", "_UDS_DATE_"]].copy()

match_idx = find_best_mri_match_for_each_uds(
    uds_df=uds_meta,
    mri_df=mri_raw,
    id_col=id_col,
    uds_date_series=uds_meta["_UDS_DATE_"],
    mri_date_col=mri_date_col,
    max_days=MRI_WINDOW_DAYS
)

mri_available_mask = match_idx.notna().astype(int)

mri_feature_cols  = [c for c in mri_raw.columns
                     if c not in [id_col, mri_date_col]]
mri_feature_table = preprocess_mri_full(mri_raw[mri_feature_cols].copy())

aligned_mri_rows = []
for idx_match in match_idx:
    if pd.isna(idx_match):
        aligned_mri_rows.append(
            np.full(len(mri_feature_cols), MISSING_VALUE, dtype=np.float32))
    else:
        aligned_mri_rows.append(
            mri_feature_table.loc[int(idx_match)].values.astype(np.float32))

mrisbm_aligned = pd.DataFrame(
    aligned_mri_rows, columns=mri_feature_cols, index=uds_meta.index)

print("\nAligned MRI shape   :", mrisbm_aligned.shape)
print("MRI available count :", int(mri_available_mask.sum()))
print("MRI missing count   :", int((1 - mri_available_mask).sum()))


# ============================================================
# SAVE — UNCHANGED (output goes to new v4 directory)
# ============================================================

subject_id_out = uds_keep[[id_col]].copy().reset_index(drop=True)
visit_meta_out = uds_keep[[id_col, "_UDS_DATE_"]].copy().reset_index(drop=True)
label_out      = uds_keep[["LABEL"]].copy().reset_index(drop=True)
mask_out       = pd.DataFrame(
    {"MRI_AVAILABLE": mri_available_mask.values.astype(int)})

uds_history    = uds_history.reset_index(drop=True)
uds_survey     = uds_survey.reset_index(drop=True)
uds_testing    = uds_testing.reset_index(drop=True)
mrisbm_aligned = mrisbm_aligned.reset_index(drop=True)

subject_id_out.to_csv(os.path.join(OUT_DIR, "subject_id.csv"),  index=False)
visit_meta_out.to_csv(os.path.join(OUT_DIR, "visit_meta.csv"),  index=False)
label_out.to_csv(     os.path.join(OUT_DIR, "label.csv"),       index=False)
mask_out.to_csv(      os.path.join(OUT_DIR, "mri_available_mask.csv"), index=False)
uds_history.to_csv(   os.path.join(OUT_DIR, "uds_history.csv"), index=False)
uds_survey.to_csv(    os.path.join(OUT_DIR, "uds_survey.csv"),  index=False)
uds_testing.to_csv(   os.path.join(OUT_DIR, "uds_testing.csv"), index=False)
mrisbm_aligned.to_csv(os.path.join(OUT_DIR, "mrisbm.csv"),      index=False)

print("\nSaved files to:", OUT_DIR)

Raw UDS shape : (185831, 1024)
Raw MRI shape : (1652, 200)
Using ID column    : NACCID
Using MRI date col : SCANDT
Eligible UDS visits after diagnosis filtering: 185831
After first-visit filter : 50259 rows (50259 unique subjects)
Raw modality dimensions before preprocessing:
  history : (50259, 66)
  survey  : (50259, 39)
  testing : (50259, 29)

Processed modality dimensions after KEEP-WIDTH preprocessing:
  history : (50259, 66)
  survey  : (50259, 39)
  testing : (50259, 29)


/home/zmoslemi/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/zmoslemi/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/zmoslemi/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)



Aligned MRI shape   : (50259, 198)
MRI available count : 1454
MRI missing count   : 48805

Saved files to: ./preprocessed_v4_firstvisit_missingMRI_keepRawDims
